# OCI / HARP2 phase identification with the 265 K all-liquid reference

Reuse the existing matched-pair cache. Default: OCI 2260 nm CER/COT, with OCI 2130 nm and HARP2 CER comparisons. The original reflectances remain untouched.

**Read `../../Documentation/algorithm_details.md` first.** This version uses a fixed-wind ocean LUT and applies that ocean surface model globally, including land/snow. Native OCI L2 microphysics are transferred by nearest time-compatible pixel, not footprint averaging. Original OCI ancillary filenames are used; unsupported near-surface profile repairs and unpopulated transmission cells are rejected. This is an exploratory implementation, not a bit-for-bit OCI retrieval reproduction.

Run the three-pair pilot first. Rerunning an interrupted stage resumes completed files. Changing scientific settings requires a new output directory.

In [ ]:
# Run only if these dependencies are missing in this environment:
# Install from the repository root in this kernel environment:
# python -m pip install -e ".[validation]"

from pathlib import Path
from datetime import date, timedelta
from dataclasses import replace
import json
import shutil
import numpy as np
import xarray as xr
import earthaccess
from pace_specpol import matching as mp
from pace_specpol.reference import Reference
from pace_specpol.validation.aggregation import IndexConfig, PhaseIndex, build_index
from pace_specpol.validation.vis_dashboard import dashboard, reference_dashboard
from pace_specpol.validation.vis_samples import plot_cached_samples
from pace_specpol.paths import load_paths
PATHS = load_paths()
from pace_specpol.liquid_reference import LiquidConfig
from pace_specpol.oci_companion import CompanionConfig, OCICompanion, discover_cloud, atomic_json
from pace_specpol.workflow import (
    cache_companions, cache_references, default_variants,
    VariantReference,
)



## Paths and initial experiment

Set `PAIR_CACHE` to the directory containing your completed `manifest.json` and matched-pair `.nc` files. Date range comes from that manifest. Set `MAX_PAIRS=None` for the complete range after inspecting the pilot. The run label changes automatically between pilot and full run.

The bulk refractive-index `.dat` file is provenance only; it is already incorporated in the supplied Mie/reflectance LUTs.

In [ ]:
PAIR_CACHE = PATHS["pair_cache"]
LUT_ROOT = PATHS["lut_root"]
WORK_ROOT = PATHS["work_root"]
MAX_PAIRS = 3             # None = all cached pairs
EXPERIMENT = "global_ocean3_265K_v1"  # change for new LUT or scientific settings
RUN = f"{EXPERIMENT}_pilot{MAX_PAIRS}" if MAX_PAIRS is not None else f"{EXPERIMENT}_full"
COMPANION_DIR = WORK_ROOT / RUN / "companions"
REFERENCE_DIR = WORK_ROOT / RUN / "references"

liquid_config = LiquidConfig(
    ms_path=str(LUT_ROOT / "LIQUID/ocean_msr_water_wspeed_3_v6.PACE.1.1.5.2026144071240.hdf"),
    phase_path=str(LUT_ROOT / "IceAndWaterPhaseFunctionData_v6.PACE.1.1.5.2026142144440.hdf"),
    transmittance_path=str(LUT_ROOT / "Transmittance_OCI.hdf"),
    cer_source="oci",
    oci_reference_band=2260,   # change to 2130 when selecting one mode
    oci_phase_policy="all",   # "liquid" = OCI phase code 2 only
    ocean_only=False,
)
liquid_config.validate()
manifest = json.loads((PAIR_CACHE / "manifest.json").read_text())
assert manifest["complete"], "Finish the original paired cache first."
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("Paired files:", len(manifest["files"]))
print("Free disk GiB:", round(shutil.disk_usage(WORK_ROOT).free / 2**30, 2))
print("Raw paired data are read-only inputs. Output:", REFERENCE_DIR)


## Discover refined OCI cloud retrievals

The new reader corrects legacy L1C observation timestamps using NASA's offset sign and searches adjacent CLD granules. Discovery includes a day of padding on either side; accepted samples still obey the original requested dates.

For AWS, `transport="earthaccess"` uses S3. `transport="https"` uses public cloud range reads and is also supported. The much smaller original meteorological files are fetched separately through OB.DAAC, using the exact names recorded in each CLD product. No replacement GEOS product is silently selected.

In [ ]:
earthaccess.login()
start = date.fromisoformat(manifest["config"]["start_date"])
end = date.fromisoformat(manifest["config"]["end_date"])
cloud_version = "3.1"
catalog_path = WORK_ROOT / f"OCI_CLD_{start}_{end}_V{cloud_version}.json"
if catalog_path.exists():
    catalog = json.loads(catalog_path.read_text())
else:
    catalog = discover_cloud(str(start-timedelta(days=1)), str(end+timedelta(days=1)), cloud_version)
    atomic_json(catalog_path, catalog)
print("Refined CLD granules, including padding:", len(catalog))

companion_config = CompanionConfig(
    cloud_version=cloud_version,
    transport="earthaccess",  # "https" is an alternative
    ancillary_cache=str(WORK_ROOT / "ancillary_downloads"),
    max_download_cache_mb=512,
    minimum_free_disk_mb=512,
    max_distance_km=3.0,
    max_time_difference_s=10.0,
)
provider = OCICompanion(
    catalog, companion_config,
    session=earthaccess.get_requests_https_session(),
)


## Stage 1: cache OCI microphysics and above-cloud water

This is the remote-data stage. It processes one paired granule at a time and writes after each. If a network or authentication error occurs, log in again, recreate `provider`, and rerun this cell. Completed companions are reused.

The original cache's erroneous date-boundary exclusions cannot be undone from cached samples alone; existing in-range samples are corrected and retained. See the README before using a legacy cache for precise boundary accounting.

In [ ]:
cache_companions(PAIR_CACHE, COMPANION_DIR, provider, max_pairs=MAX_PAIRS)


## Stage 2: all-liquid references (local computation)

Both OCI modes use their matching COT/CER pair. HARP2 modes replace CER only. The same atmospheric correction is used throughout. An ice-derived OCI CER is treated explicitly as a same-numerical-radius liquid counterfactual; out-of-range CER is rejected, not clamped.

The dictionary below controls which variants are computed. The common population is the intersection valid in every listed variant. For a two-mode comparison, retain only those two dictionary entries before creating a new reference directory.

In [ ]:
variants = default_variants(liquid_config)
# Example: only compare CER sources with OCI 2260 COT:
# variants = {k: variants[k] for k in ("oci_2260", "harp2_2260")}

cache_references(COMPANION_DIR, REFERENCE_DIR, variants)


## Inspect exclusions before interpreting phase maps

Reason counts overlap: one sample can fail more than one check. Excluded pixels are missing, never relabeled as ice or liquid. Differences in each mode's valid population must not be interpreted as phase changes.

In [ ]:
report = json.loads((REFERENCE_DIR / "manifest.json").read_text())
for variant in variants:
    records = [r["variants"][variant] for r in report["records"]]
    print(variant, "valid:", sum(r["valid"] for r in records),
          "ice CER used as liquid:", sum(r["ice_cer_as_liquid_valid"] for r in records))
    print({reason: sum(r["reason_counts"][reason] for r in records)
           for reason in records[0]["reason_counts"]})
print("Common valid:", sum(r["common_valid"] for r in report["records"]))
print("Free disk GiB:", round(shutil.disk_usage(WORK_ROOT).free / 2**30, 2))


## Interactive comparison

Choose CER/COT mode, sample population, and ratio space; click **Show selection**. The first selection builds an on-disk sparse index. Later use reopens it. Threshold changes inside the dashboard need no remote reads or LUT recomputation.

1.27 remains a provisional threshold, not a newly calibrated OCI mixed-phase boundary. Compare the common population to isolate reference changes. The maps show modal phase plus mean LI and mean ratio; rare phases can be hidden by the modal map.

In [ ]:
reference_dashboard(
    REFERENCE_DIR,
    index_config=IndexConfig(resolution=0.1, ratio_min=0.5, ratio_max=3.5, ratio_step=0.01),
    threshold=1.27,
    ratio_clim=(0.5, 2.0),
    li_clim=(-0.5, 4.0),
    output_dir=str(PATHS["export_root"] / RUN),
)


## Optional full-resolution export for a selected experiment

Exports retain class counts and fractions as well as dominant phase. Use a descriptive filename. This cell reads the prepared reference cache only.

In [ ]:
SELECTED_VARIANT = "oci_2260"  # "oci_2130", "harp2_2260", "harp2_2130"
SELECTED_POPULATION = "own"    # "common"
SELECTED_THRESHOLD = 1.27
# Uncomment to build/export:
# index = build_index(REFERENCE_DIR,
#     reference=VariantReference(SELECTED_VARIANT, SELECTED_POPULATION),
#     config=IndexConfig(resolution=0.1))
# output = WORK_ROOT / RUN / f"{SELECTED_VARIANT}_{SELECTED_POPULATION}_phase.nc"
# index.export(output, threshold=SELECTED_THRESHOLD)
